# Expressions - Python

All 2 Python examples from [docs/expression.md](https://platob.github.io/yggdryl/expression/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

## The four stages

In [ ]:
from decimal import Decimal

from yggdryl import Expression, Field

schema = Field("trades", "struct<ccy:utf8,price:decimal(9,2),size:bigint>", False)
filter = Expression("ccy = 'EUR' and price > 100")

assert str(filter) == "ccy = 'EUR' and price > 100"
assert filter.columns() == ["ccy", "price"]

bound = filter.bind(schema)
assert str(bound.expression) == "ccy = 'EUR' and price > decimal128(9,2) '100.00'"

# A row is a sequence in schema order, or a mapping of column to value.
# The price is a `Decimal`, because the column is exact and so is the
# comparison: a float here would be a different number.
assert bound.matches(["EUR", Decimal("150.00"), 5])
assert bound.matches({"ccy": "EUR", "price": Decimal("150.00"), "size": 5})
assert not bound.matches({"ccy": "USD", "price": Decimal("150.00"), "size": 5})

## `&holder.*`: asking about the file

In [ ]:
import tempfile
from pathlib import Path

from yggdryl import IOBase

with tempfile.TemporaryDirectory() as root:
    for year in ("2024", "2025"):
        leaf = Path(root) / f"year={year}"
        leaf.mkdir()
        (leaf / "part-0.parquet").write_bytes(b"")

    lake = IOBase(root)
    matched = lake.children_matching("&holder.partition['year'] = '2024'")
    assert matched
    assert all("year=2024" in str(entry.url) for entry in matched)